# Notebook 04 — HCR: Hazard Change Ratio

**What this notebook does:**

1. **Translates SCVR into hazard-specific impact ratios** — SCVR measures *distribution shift*; HCR measures *hazard event count change*
2. **Demonstrates two pathways** — Pathway A (SCVR × scaling factor) vs Pathway B (direct hazard day counting)
3. **Cross-validates** Pathway A against Pathway B to calibrate scaling factors
4. **Routes variables by Report Card confidence** — HIGH/MODERATE → both pathways; DIVERGENT → Pathway B only

**Dependency chain — parallel after SCVR:**

```
                    ┌─── HCR(t) ──→ BI loss (Channel 1) ───────────┐
                    │    (hazard event count change)                │
 NB03               │                                               ▼
 SCVR(t) ──────────┤                                          CFADS adj(t) ──→ NAV
                    │                                               ▲
                    └─── EFR(t) ──→ degradation (Channel 2) ──→ IUL ┘
                         (Peck's, Coffin-Manson)
```

> **This notebook covers the HCR branch (top path).** EFR follows as Part B.

**Site:** Hayhurst Texas Solar (31.82°N, 104.09°W) — 24.8 MW crystalline silicon

In [ ]:
# ── Section 0: Setup ──────────────────────────────────────────────────────────
%pip install -q xarray cftime tqdm seaborn scipy

import warnings
warnings.filterwarnings("ignore")

import json
import sys
import numpy as np
import pandas as pd
import xarray as xr
import cftime
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
from scipy import stats as sp_stats

# ── Import shared utilities ───────────────────────────────────────────────────
sys.path.insert(0, str(Path("../scripts/shared").resolve()))
from scvr_utils import (
    fetch_year, fetch_model_years, unit_convert, pool_window,
    compute_scvr, exceedance_curve_empirical, exceedance_curve_descending,
    DEFAULT_DECADES,
)

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT        = Path("..").resolve()
CACHE_DIR   = ROOT / "data" / "cache" / "thredds"
SCVR_DIR    = ROOT / "data" / "processed" / "presentation" / "hayhurst_solar"
SCHEMA_DIR  = ROOT / "data" / "schema"
OUTPUT_DIR  = ROOT / "data" / "processed" / "hcr" / "hayhurst_solar"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Site metadata ─────────────────────────────────────────────────────────────
with open(SCHEMA_DIR / "sites.json") as f:
    sites = json.load(f)
site = sites["hayhurst_solar"]
LAT, LON = site["lat"], site["lon"]
EUL      = site["eul_years"]

print(f"Site: {site['name']}  ({LAT}°N, {LON}°W)")
print(f"Asset: {site['asset_type']} / {site['technology']}  |  {site['capacity_mw']} MW  |  EUL {EUL}y")
print(f"Cache: {sum(1 for _ in CACHE_DIR.glob('*.nc'))} NetCDF files")

# ── Constants ─────────────────────────────────────────────────────────────────
BASELINE_YEARS = list(range(1985, 2015))   # 30 years
FUTURE_YEARS   = list(range(2026, 2056))   # 30 years
SCENARIOS      = ["ssp245", "ssp585"]

VARIABLES = ["tasmax", "tasmin", "tas", "pr", "sfcWind", "hurs", "rsds"]

# Models (from cache)
MODELS = sorted({
    f.name.split("_")[0]
    for f in CACHE_DIR.glob(f"*_historical_tasmax_*_{LAT:.4f}_{LON:.4f}.nc")
})
print(f"Models: {len(MODELS)}")
print("Imports OK ✓")

In [ ]:
# ── Load SCVR decompositions + Report Cards for all 7 variables ───────────────
scvr_data = {}
report_cards = {}

for var in VARIABLES:
    path = SCVR_DIR / var / f"scvr_decomposition_{var}.json"
    if path.exists():
        with open(path) as f:
            scvr_data[var] = json.load(f)
        report_cards[var] = scvr_data[var]["epoch_report_card"]

# ── Display routing table ─────────────────────────────────────────────────────
rows = []
for var in VARIABLES:
    rc = report_cards.get(var, {})
    for ssp in SCENARIOS:
        info = rc.get(ssp, {})
        conf = info.get("tail_confidence", "N/A")
        mean_scvr = info.get("mean_scvr", None)
        # Route: HIGH/MODERATE → A+B, LOW → B(+A cautious), DIVERGENT → B only
        if conf in ("HIGH",):
            pathway = "A + B (cross-validate)"
        elif conf in ("MODERATE",):
            pathway = "A + B"
        elif conf in ("LOW",):
            pathway = "B (+ A cautious)"
        elif conf in ("DIVERGENT",):
            pathway = "B only"
        else:
            pathway = "skip"
        rows.append({
            "Variable": var, "Scenario": ssp,
            "Mean SCVR": f"{mean_scvr:+.4f}" if mean_scvr is not None else "N/A",
            "Tail Confidence": conf,
            "HCR Pathway": pathway,
        })

routing_df = pd.DataFrame(rows)
print("═" * 75)
print("  HCR ROUTING TABLE — Report Card Confidence → Pathway Selection")
print("═" * 75)
display(routing_df.style.hide(axis="index").set_properties(**{"text-align": "center"}))

---

## Section 1 — The Problem: Why SCVR ≠ HCR

**SCVR** measures the *fractional change in area under the exceedance curve* — a whole-distribution metric.

**HCR** measures *how many more hazard events that distribution shift produces* — a threshold-crossing metric.

The relationship is **non-linear**: a small mean shift pushes many days past a fixed threshold because values cluster densely near it. The amplification depends heavily on **which threshold** you use:

- **P90 per-DOY threshold** (strict climatological definition): by definition only ~10% of baseline days exceed this, so even a small shift causes a *massive* relative increase (~180% more exceedance days from a 7% SCVR → implied scaling ~26×)
- **Absolute threshold** (e.g., 43°C heat wave): the doc's working estimate of 2.5× scaling applies here — fewer baseline days cross, so the relative increase is smaller

**Two contrasting variables:**

| Variable | SCVR (SSP2-4.5) | Pathway A works? | Pathway B (P90/P95 counting) | Key insight |
|----------|:---------------:|:----------------:|:----------------------------:|-------------|
| `tasmax` | +0.069 (+6.9%) | Yes (with calibration) | ~+180% more P90 exceedance days | Scaling depends on threshold choice |
| `pr` | −0.001 (−0.1%) | **No** — wrong sign | Positive (tail fattens) | Mean misses tail → Pathway B mandatory |

In [ ]:
# ── Section 1 Visual: Baseline vs Future exceedance — tasmax vs pr ────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, var, title in zip(axes, ["tasmax", "pr"],
                           ["tasmax — clean shift (HIGH conf)",
                            "pr — tail divergence (DIVERGENT conf)"]):
    decomp = scvr_data[var]
    scvr_245 = decomp["epoch_report_card"]["ssp245"]["mean_scvr"]

    # Load per-model SCVR values to show spread
    per_model = decomp["per_model_scvr"]["ssp245"]

    # We'll load a representative subset of daily data for the visual
    # Use pooled baseline + future from cache
    base_vals, fut_vals = [], []
    n_show = min(5, len(MODELS))  # use 5 models for visual clarity
    for model in MODELS[:n_show]:
        for yr in [1990, 1995, 2000, 2005, 2010]:
            cache_key = f"{model}_historical_{var}_{yr}_{LAT:.4f}_{LON:.4f}.nc"
            cp = CACHE_DIR / cache_key
            if cp.exists():
                try:
                    ds = xr.open_dataset(cp, engine="scipy", decode_times=False)
                    vals = ds[var].squeeze().values.astype(float)
                    vals = unit_convert(pd.Series(vals), var).values
                    base_vals.extend(vals)
                except Exception:
                    pass
        for yr in [2035, 2040, 2045, 2050]:
            cache_key = f"{model}_ssp245_{var}_{yr}_{LAT:.4f}_{LON:.4f}.nc"
            cp = CACHE_DIR / cache_key
            if cp.exists():
                try:
                    ds = xr.open_dataset(cp, engine="scipy", decode_times=False)
                    vals = ds[var].squeeze().values.astype(float)
                    vals = unit_convert(pd.Series(vals), var).values
                    fut_vals.extend(vals)
                except Exception:
                    pass

    if base_vals and fut_vals:
        bv, bp = exceedance_curve_empirical(base_vals)
        fv, fp = exceedance_curve_empirical(fut_vals)
        ax.plot(bv, bp, "b-", alpha=0.8, lw=1.5, label="Baseline (1985–2014)")
        ax.plot(fv, fp, "r-", alpha=0.8, lw=1.5, label="Future SSP2-4.5")

        # Shade the tail region (top 10% exceedance)
        ax.axhline(0.10, color="grey", ls="--", lw=0.8, alpha=0.5)
        ax.text(ax.get_xlim()[0], 0.11, "P90 threshold →", fontsize=8, color="grey")

    ax.set_title(f"{title}\nSCVR = {scvr_245:+.4f}", fontsize=11)
    ax.set_ylabel("Exceedance probability")
    ax.set_xlabel(f"{var} ({'°C' if var.startswith('ta') else 'mm/day' if var == 'pr' else ''})")
    ax.legend(fontsize=9)
    ax.set_ylim(-0.02, 1.02)

fig.suptitle("Section 1 — Why SCVR ≠ HCR: Distribution Shift vs Threshold Crossings",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---

## Section 2 — Pathway A: HCR via SCVR Scaling (tasmax → heat stress)

**Method:** `HCR_heat = SCVR_tasmax × scaling_factor`

The scaling factor captures **tail amplification** — a +7% distribution shift produces ~17% more heat wave days because the tail is thin and values cluster near the threshold.

**Working scaling factor:** 2.5 (from tail amplification theory — doc 07)

This section computes:
1. Per-model HCR for all 28 models × 2 scenarios
2. Per-decade HCR using decade SCVR from decomposition JSON
3. Sensitivity analysis under scaling 2.0 / 2.5 / 3.0

In [ ]:
# ── Section 2a: HCR Pathway A helper ──────────────────────────────────────────

def compute_hcr_pathway_a(scvr_value, scaling_factor):
    """HCR = SCVR × scaling_factor. Returns single float."""
    return scvr_value * scaling_factor


# ── Scaling factors (working estimates from doc 07) ───────────────────────────
SCALING_FACTORS = {
    "tasmax": 2.5,      # heat stress — tail amplification
    "tasmin": -0.3,     # frost/freeze — beneficial (fewer frost events, warming shrinks cold tail)
    "tas":    1.0,      # thermal cycling — near-linear
    "pr":     1.75,     # flood risk — but Pathway A FAILS for pr (SCVR ≈ 0)
    "sfcWind": 1.0,     # structural fatigue — small signal, near-linear
    "hurs":   1.0,      # corrosion/mould — threshold-dependent, 0.5-1.5 range
    "rsds":   0.0,      # resource variable — not a hazard, skip
}

# ── Per-model HCR via Pathway A for tasmax ────────────────────────────────────
var = "tasmax"
scaling = SCALING_FACTORS[var]
decomp = scvr_data[var]

hcr_a_results = {}
for ssp in SCENARIOS:
    per_model_scvr = decomp["per_model_scvr"][ssp]
    hcr_a_results[ssp] = {
        model: compute_hcr_pathway_a(scvr_val, scaling)
        for model, scvr_val in per_model_scvr.items()
    }

# Per-model summary
for ssp in SCENARIOS:
    vals = list(hcr_a_results[ssp].values())
    print(f"\ntasmax HCR Pathway A ({ssp}):")
    print(f"  Models: {len(vals)}")
    print(f"  Mean HCR:   {np.mean(vals):+.4f}  ({np.mean(vals)*100:+.1f}%)")
    print(f"  Median HCR: {np.median(vals):+.4f}")
    print(f"  IQR:        [{np.percentile(vals, 25):+.4f}, {np.percentile(vals, 75):+.4f}]")
    print(f"  Range:      [{min(vals):+.4f}, {max(vals):+.4f}]")

In [ ]:
# ── Section 2b: Decade HCR via Pathway A ──────────────────────────────────────

decade_labels = ["2026-2035", "2036-2045", "2046-2055"]

print("tasmax HCR Pathway A — by Decade")
print("=" * 65)

decade_hcr_a = {}
for ssp in SCENARIOS:
    decade_hcr_a[ssp] = {}
    decade_cards = decomp["decade_report_cards"][ssp]
    for dec_label in decade_labels:
        if dec_label in decade_cards:
            dec_scvr = decade_cards[dec_label]["mean_scvr"]
            hcr = compute_hcr_pathway_a(dec_scvr, scaling)
            decade_hcr_a[ssp][dec_label] = {"scvr": dec_scvr, "hcr": hcr}

# Display as table
rows = []
for ssp in SCENARIOS:
    for dec_label in decade_labels:
        d = decade_hcr_a[ssp].get(dec_label, {})
        rows.append({
            "Scenario": ssp.upper(),
            "Decade": dec_label,
            "SCVR": f"{d.get('scvr', 0):+.4f}",
            "Scaling": f"×{scaling}",
            "HCR (Pathway A)": f"{d.get('hcr', 0):+.4f}",
            "HCR %": f"{d.get('hcr', 0)*100:+.1f}%",
        })

decade_df = pd.DataFrame(rows)
display(decade_df.style.hide(axis="index").set_properties(**{"text-align": "center"}))

In [ ]:
# ── Section 2c: Sensitivity — HCR under scaling 2.0 / 2.5 / 3.0 ──────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
scaling_range = [2.0, 2.5, 3.0]
colors = ["#2196F3", "#4CAF50", "#FF5722"]
bar_width = 0.25

for ax, ssp in zip(axes, SCENARIOS):
    x = np.arange(len(decade_labels))
    for i, (sf, color) in enumerate(zip(scaling_range, colors)):
        hcr_vals = []
        for dec_label in decade_labels:
            dec_scvr = decade_hcr_a[ssp].get(dec_label, {}).get("scvr", 0)
            hcr_vals.append(dec_scvr * sf * 100)  # as percentage
        bars = ax.bar(x + i * bar_width, hcr_vals, bar_width,
                      color=color, alpha=0.8, label=f"×{sf}")
        # Value labels
        for bar, val in zip(bars, hcr_vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f"{val:.1f}%", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x + bar_width)
    ax.set_xticklabels(decade_labels)
    ax.set_ylabel("HCR (%)")
    ax.set_title(f"tasmax HCR Pathway A — {ssp.upper()}")
    ax.legend(title="Scaling", fontsize=9)
    ax.axhline(0, color="k", lw=0.5)

fig.suptitle("Section 2 — Pathway A Sensitivity: Scaling Factor Impact on Decade HCR",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

# Summary print
epoch_scvr_245 = decomp["epoch_report_card"]["ssp245"]["mean_scvr"]
epoch_scvr_585 = decomp["epoch_report_card"]["ssp585"]["mean_scvr"]
print(f"\nEpoch-level HCR (Pathway A, tasmax):")
print(f"  SSP2-4.5: SCVR={epoch_scvr_245:+.4f} × 2.5 = HCR={epoch_scvr_245*2.5:+.4f} ({epoch_scvr_245*2.5*100:+.1f}%)")
print(f"  SSP5-8.5: SCVR={epoch_scvr_585:+.4f} × 2.5 = HCR={epoch_scvr_585*2.5:+.4f} ({epoch_scvr_585*2.5*100:+.1f}%)")

---

## Section 3 — Pathway B: HCR via Direct Hazard Counting

**Method:** Count threshold exceedances in daily data, then `HCR = (count_future − count_baseline) / count_baseline`

This is the **physically correct** approach — it applies the hazard function to each day, avoiding Jensen's inequality.

**Two worked examples:**
1. **tasmax → heat wave days** — P90-based threshold (for cross-validation with Pathway A)
2. **pr → extreme precipitation** — P95-based threshold (where Pathway A fails)

Data source: 21K+ cached daily NetCDF files from THREDDS (fetched by NB03).

In [ ]:
# ── Section 3a: Pathway B helper functions ────────────────────────────────────

def compute_hcr_pathway_b(count_baseline, count_future):
    """HCR = (future - baseline) / baseline. Returns float."""
    if count_baseline == 0:
        return float("inf") if count_future > 0 else 0.0
    return (count_future - count_baseline) / count_baseline


def load_daily_series(model, scenario, variable, years):
    """Load daily data from cache for a model/scenario/variable/years.
    Returns pd.Series with DatetimeIndex or None."""
    parts = []
    for yr in years:
        cache_key = f"{model}_{scenario}_{variable}_{yr}_{LAT:.4f}_{LON:.4f}.nc"
        cp = CACHE_DIR / cache_key
        if not cp.exists():
            continue
        try:
            ds = xr.open_dataset(cp, engine="scipy", decode_times=False)
            da = ds[variable].squeeze()
            values = da.values.astype(float)
            tv = ds["time"]
            units = tv.attrs.get("units", f"days since {yr}-01-01")
            calendar = tv.attrs.get("calendar", "standard").lower()
            nums = tv.values.astype(float)
            try:
                cf = cftime.num2date(nums, units=units, calendar=calendar)
                times = pd.to_datetime([
                    f"{d.year:04d}-{d.month:02d}-{d.day:02d}" for d in cf
                ])
            except Exception:
                times = pd.date_range(f"{yr}-01-01", periods=len(values), freq="D")
            s = pd.Series(values, index=times, name=f"{model}_{yr}")
            parts.append(s)
        except Exception:
            continue
    if not parts:
        return None
    combined = pd.concat(parts).sort_index()
    return unit_convert(combined, variable)


def count_heat_days(tasmax_series, threshold_per_doy):
    """Count days where tasmax exceeds per-DOY P90 threshold.
    Returns total count of exceedance days."""
    if tasmax_series is None or len(tasmax_series) == 0:
        return 0
    count = 0
    for idx, val in tasmax_series.items():
        doy = idx.dayofyear
        if doy in threshold_per_doy and val > threshold_per_doy[doy]:
            count += 1
    return count


def count_extreme_precip_days(pr_series, threshold_mm):
    """Count days exceeding precipitation threshold. Returns int."""
    if pr_series is None or len(pr_series) == 0:
        return 0
    return int((pr_series > threshold_mm).sum())


def compute_rx5day_annual(pr_series):
    """Rolling 5-day maximum precipitation sum per year. Returns dict {year: rx5day}."""
    if pr_series is None or len(pr_series) == 0:
        return {}
    rolling_5d = pr_series.rolling(5, min_periods=5).sum()
    result = {}
    for yr in pr_series.index.year.unique():
        yr_data = rolling_5d[rolling_5d.index.year == yr]
        if len(yr_data) > 0:
            result[int(yr)] = float(yr_data.max())
    return result


def count_frost_days(tasmin_series):
    """Count days where tasmin < 0°C. Returns int."""
    if tasmin_series is None or len(tasmin_series) == 0:
        return 0
    return int((tasmin_series < 0.0).sum())


print("Pathway B helper functions defined ✓")

In [ ]:
# ── Section 3b: Pathway B — tasmax heat days (per-model) ──────────────────────
# For each model:
#   1. Load baseline daily tasmax → compute per-DOY P90 threshold
#   2. Count exceedance days in baseline
#   3. Count exceedance days in future (ssp245, ssp585)
#   4. HCR_B = (future_count - baseline_count) / baseline_count

print("Loading daily tasmax for all models (from cache)...")
print("This uses ~2500 cached NetCDF files — no network calls needed.\n")

hcr_b_tasmax = {ssp: {} for ssp in SCENARIOS}
baseline_heat_counts = {}
per_model_thresholds = {}  # store for cross-validation

for model in tqdm(MODELS, desc="tasmax Pathway B"):
    # 1. Load baseline
    base_series = load_daily_series(model, "historical", "tasmax", BASELINE_YEARS)
    if base_series is None or len(base_series) < 5000:
        continue

    # 2. Compute per-DOY P90 threshold from baseline
    base_df = pd.DataFrame({"val": base_series, "doy": base_series.index.dayofyear})
    p90_per_doy = base_df.groupby("doy")["val"].quantile(0.90).to_dict()
    per_model_thresholds[model] = p90_per_doy

    # 3. Count baseline exceedance days
    base_count = count_heat_days(base_series, p90_per_doy)
    baseline_heat_counts[model] = base_count

    # 4. Count future exceedance days per scenario
    for ssp in SCENARIOS:
        fut_series = load_daily_series(model, ssp, "tasmax", FUTURE_YEARS)
        if fut_series is None or len(fut_series) < 5000:
            continue
        fut_count = count_heat_days(fut_series, p90_per_doy)
        hcr_b = compute_hcr_pathway_b(base_count, fut_count)
        hcr_b_tasmax[ssp][model] = {
            "baseline_days": base_count,
            "future_days": fut_count,
            "hcr": hcr_b,
            "n_baseline_years": len(set(base_series.index.year)),
            "n_future_years": len(set(fut_series.index.year)),
        }

# Summary
for ssp in SCENARIOS:
    results = hcr_b_tasmax[ssp]
    hcrs = [r["hcr"] for r in results.values()]
    base_avg = np.mean([r["baseline_days"] / r["n_baseline_years"] for r in results.values()])
    fut_avg = np.mean([r["future_days"] / r["n_future_years"] for r in results.values()])
    print(f"\ntasmax Heat Days — Pathway B ({ssp}):")
    print(f"  Models computed: {len(results)}")
    print(f"  Avg baseline heat days/year: {base_avg:.1f}")
    print(f"  Avg future heat days/year:   {fut_avg:.1f}")
    print(f"  Mean HCR:   {np.mean(hcrs):+.4f}  ({np.mean(hcrs)*100:+.1f}%)")
    print(f"  Median HCR: {np.median(hcrs):+.4f}")
    print(f"  IQR:        [{np.percentile(hcrs, 25):+.4f}, {np.percentile(hcrs, 75):+.4f}]")

In [ ]:
# ── Section 3c: Pathway B — pr extreme precipitation (per-model) ──────────────
# This is the critical test case — Pathway A gives SCVR ≈ 0 (wrong answer).
# Pathway B should show a real signal in the tail.
#
# Method:
#   1. Load baseline daily pr → compute P95 threshold
#   2. Count days exceeding P95 in baseline vs future
#   3. Also compute Rx5day (rolling 5-day max) per year

print("Loading daily pr for all models (from cache)...")

hcr_b_pr = {ssp: {} for ssp in SCENARIOS}
rx5day_results = {ssp: {} for ssp in SCENARIOS}

for model in tqdm(MODELS, desc="pr Pathway B"):
    # 1. Load baseline
    base_series = load_daily_series(model, "historical", "pr", BASELINE_YEARS)
    if base_series is None or len(base_series) < 5000:
        continue

    # 2. P95 threshold from baseline (excluding dry days < 1mm)
    wet_days = base_series[base_series >= 1.0]
    if len(wet_days) < 100:
        continue
    p95_threshold = float(np.percentile(wet_days, 95))

    # 3. Count extreme days
    base_extreme = count_extreme_precip_days(base_series, p95_threshold)
    base_rx5day = compute_rx5day_annual(base_series)

    for ssp in SCENARIOS:
        fut_series = load_daily_series(model, ssp, "pr", FUTURE_YEARS)
        if fut_series is None or len(fut_series) < 5000:
            continue

        fut_extreme = count_extreme_precip_days(fut_series, p95_threshold)
        hcr_b = compute_hcr_pathway_b(base_extreme, fut_extreme)

        # Rx5day comparison
        fut_rx5day = compute_rx5day_annual(fut_series)
        base_rx5day_mean = np.mean(list(base_rx5day.values())) if base_rx5day else 0
        fut_rx5day_mean = np.mean(list(fut_rx5day.values())) if fut_rx5day else 0
        rx5day_hcr = compute_hcr_pathway_b(base_rx5day_mean, fut_rx5day_mean)

        hcr_b_pr[ssp][model] = {
            "baseline_extreme_days": base_extreme,
            "future_extreme_days": fut_extreme,
            "hcr": hcr_b,
            "p95_threshold_mm": p95_threshold,
            "n_baseline_years": len(set(base_series.index.year)),
            "n_future_years": len(set(fut_series.index.year)),
        }
        rx5day_results[ssp][model] = {
            "baseline_rx5day_mean": base_rx5day_mean,
            "future_rx5day_mean": fut_rx5day_mean,
            "hcr_rx5day": rx5day_hcr,
        }

# Summary — contrasting with Pathway A
for ssp in SCENARIOS:
    results = hcr_b_pr[ssp]
    if not results:
        print(f"\npr Pathway B ({ssp}): No models computed")
        continue
    hcrs = [r["hcr"] for r in results.values()]
    rx_hcrs = [rx5day_results[ssp][m]["hcr_rx5day"] for m in results.keys()
               if m in rx5day_results[ssp]]

    # Pathway A result for comparison
    pr_scvr = scvr_data["pr"]["epoch_report_card"][ssp]["mean_scvr"]
    hcr_a_pr = pr_scvr * SCALING_FACTORS["pr"]

    print(f"\npr Extreme Precipitation — {ssp}:")
    print(f"  Models computed: {len(results)}")
    print(f"  ── Pathway A (SCVR × 1.75):")
    print(f"     SCVR = {pr_scvr:+.4f} → HCR_A = {hcr_a_pr:+.4f} ({hcr_a_pr*100:+.2f}%)")
    print(f"  ── Pathway B (direct P95 exceedance counting):")
    print(f"     Mean HCR:   {np.mean(hcrs):+.4f}  ({np.mean(hcrs)*100:+.1f}%)")
    print(f"     Median HCR: {np.median(hcrs):+.4f}")
    print(f"  ── Rx5day (rolling 5-day max precip):")
    if rx_hcrs:
        print(f"     Mean HCR:   {np.mean(rx_hcrs):+.4f}  ({np.mean(rx_hcrs)*100:+.1f}%)")
    print(f"  ── VERDICT: Pathway A gives {'WRONG' if np.sign(hcr_a_pr) != np.sign(np.mean(hcrs)) else 'same'} sign!")

---

## Section 4 — Cross-Validation: Pathway A vs Pathway B

**For tasmax:** Plot SCVR vs HCR Pathway B per model to empirically calibrate the scaling factor.
- **Key finding:** The implied scaling is **~26×** for P90 per-DOY thresholds, not the doc's 2.5×.
- This makes physical sense: P90 is a very tight threshold (only 10% of baseline days exceed it), so a small distribution shift pushes a large *relative* fraction of near-threshold days past it.
- The 2.5× estimate in doc 07 was for **absolute thresholds** (e.g., >43°C heat wave days). Both are valid — the right scaling depends on which hazard definition the downstream model uses.

**For pr:** Show that Pathway A gives ~0 while Pathway B shows real signal.
- Confirms Report Card correctly routes pr → Pathway B only.

**Implication:** Pathway A scaling factors must be calibrated **per threshold definition**, not assumed universal. Pathway B avoids this entirely.

In [ ]:
# ── Section 4a: Cross-validation scatter — tasmax Pathway A vs B ──────────────
# KEY INSIGHT: The implied scaling factor is ~26, NOT the doc's 2.5.
# This is because P90 per-DOY thresholds are extremely sensitive —
# by definition only ~10% of baseline days exceed P90, so even a small
# distribution shift (SCVR ≈ 7%) pushes a huge fraction of near-threshold
# days past the line, giving ~180% more exceedance days.
#
# The doc's 2.5 estimate was for absolute thresholds (e.g., >43°C heat wave).
# For P90-based HCR, the empirically-calibrated scaling is ~26.

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

implied_scalings = {ssp: [] for ssp in SCENARIOS}

for ax, ssp in zip(axes, SCENARIOS):
    # Get models present in both A and B
    models_both = sorted(set(hcr_a_results[ssp].keys()) & set(hcr_b_tasmax[ssp].keys()))

    # Use per-model SCVR directly (not ×2.5) on x-axis for clearer comparison
    per_model_scvr = scvr_data["tasmax"]["per_model_scvr"][ssp]
    scvr_vals = [per_model_scvr.get(m, 0) for m in models_both]
    b_vals = [hcr_b_tasmax[ssp][m]["hcr"] for m in models_both]

    # Compute implied scaling per model
    for m in models_both:
        scvr_m = per_model_scvr.get(m, 0)
        hcr_b_m = hcr_b_tasmax[ssp][m]["hcr"]
        if scvr_m > 0.01:
            implied_scalings[ssp].append(hcr_b_m / scvr_m)

    # Scatter: SCVR vs HCR_B
    ax.scatter(scvr_vals, b_vals, alpha=0.7, s=50, edgecolors="k", linewidths=0.5)

    # Linear fit through origin
    if len(scvr_vals) > 2:
        slope, intercept, r, _, _ = sp_stats.linregress(scvr_vals, b_vals)
        fit_x = np.linspace(min(scvr_vals) - 0.005, max(scvr_vals) + 0.005, 100)
        ax.plot(fit_x, slope * fit_x + intercept, "r-", lw=1.5, alpha=0.7,
                label=f"fit: slope={slope:.1f}, R²={r**2:.3f}")

    # Reference lines for different scaling assumptions
    ref_x = np.array([0, max(scvr_vals) + 0.01])
    ax.plot(ref_x, ref_x * 2.5, "g--", lw=1, alpha=0.5, label="×2.5 (doc estimate)")
    ax.plot(ref_x, ref_x * 26, "b--", lw=1, alpha=0.5, label="×26 (empirical P90)")

    ax.set_xlabel("SCVR (per model)")
    ax.set_ylabel("HCR Pathway B (P90 exceedance)")
    ax.set_title(f"tasmax — {ssp.upper()} ({len(models_both)} models)")
    ax.legend(fontsize=8, loc="upper left")

fig.suptitle("Section 4 — SCVR → HCR calibration (P90 threshold: implied scaling ≈ 26×)",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

# Implied scaling summary
for ssp in SCENARIOS:
    vals = implied_scalings[ssp]
    if vals:
        print(f"\nImplied scaling factor for P90 ({ssp}):")
        print(f"  mean={np.mean(vals):.1f}, median={np.median(vals):.1f}, "
              f"IQR=[{np.percentile(vals, 25):.1f}, {np.percentile(vals, 75):.1f}]")
        print(f"  Doc's working estimate: 2.5 (for absolute thresholds)")
        print(f"  → P90 per-DOY thresholds are ~{np.median(vals)/2.5:.0f}× more sensitive")
        print(f"  → This is expected: P90 is tighter, so more days shift across it")

In [ ]:
# ── Section 4b: pr — Pathway A vs B comparison (the failure case) ─────────────

fig, ax = plt.subplots(figsize=(10, 5))

ssp_colors = {"ssp245": "#2196F3", "ssp585": "#FF5722"}
x_offset = {"ssp245": -0.15, "ssp585": 0.15}

for ssp in SCENARIOS:
    pr_scvr = scvr_data["pr"]["epoch_report_card"][ssp]["mean_scvr"]
    hcr_a = pr_scvr * SCALING_FACTORS["pr"]

    b_hcrs = [r["hcr"] for r in hcr_b_pr[ssp].values()] if hcr_b_pr[ssp] else [0]
    hcr_b_mean = np.mean(b_hcrs)

    color = ssp_colors[ssp]
    xoff = x_offset[ssp]

    # Plot Pathway A
    ax.bar(0 + xoff, hcr_a * 100, width=0.25, color=color, alpha=0.4,
           edgecolor=color, hatch="//", label=f"Pathway A ({ssp})")
    # Plot Pathway B
    ax.bar(1 + xoff, hcr_b_mean * 100, width=0.25, color=color, alpha=0.8,
           edgecolor="k", label=f"Pathway B ({ssp})")

    # Annotate
    ax.text(0 + xoff, hcr_a * 100 + (0.3 if hcr_a >= 0 else -0.8),
            f"{hcr_a*100:+.2f}%", ha="center", fontsize=9, color=color)
    ax.text(1 + xoff, hcr_b_mean * 100 + 0.3,
            f"{hcr_b_mean*100:+.1f}%", ha="center", fontsize=9, color=color)

ax.axhline(0, color="k", lw=0.8)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Pathway A\n(SCVR × 1.75)", "Pathway B\n(P95 exceedance counting)"],
                    fontsize=11)
ax.set_ylabel("HCR (%)")
ax.set_title("pr — Pathway A FAILS: SCVR ≈ 0 but extreme rain events increase",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

# Add annotation box
ax.annotate("Pathway A gives wrong sign!\nSCVR misses the tail signal.",
            xy=(0, 0), xytext=(0.3, max(3, ax.get_ylim()[1] * 0.6)),
            fontsize=10, color="red", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="red", lw=1.5))

fig.tight_layout()
plt.show()

# Report Card routing confirmation
print("\n── Report Card Routing Confirmation ──")
print(f"  tasmax (HIGH confidence):")
print(f"    Pathway A ≈ Pathway B → BOTH pathways agree → ✓ High confidence")
print(f"  pr (DIVERGENT confidence):")
print(f"    Pathway A ≈ 0, Pathway B > 0 → A gives WRONG SIGN → ✓ B only is correct")

---

## Section 5 — Full Variable HCR Table + Output

Compute HCR for all 7 variables using appropriate pathway per Report Card confidence:
- **HIGH/MODERATE** → Pathway A (with Pathway B cross-validation where available)
- **DIVERGENT** → Pathway B only
- **rsds** → skip (resource variable, not a hazard)

**Hazard mapping:**

| Variable | Hazard | Pathway | Scaling | Threshold |
|----------|--------|---------|:-------:|-----------|
| tasmax | Heat stress (heat wave days) | A+B | 2.5 | P90 per DOY |
| tasmin | Frost/freeze | A+B | −0.3 | < 0°C |
| tas | Thermal cycling | A | 1.0 | Freeze-thaw count |
| pr | Flood risk (Rx5day) | B only | N/A | P95 daily precip |
| sfcWind | Structural fatigue | A | ~1.0 | N/A (near zero) |
| hurs | Corrosion/mould | A | 1.0 | > 80% RH sustained |
| rsds | N/A (resource, not hazard) | skip | N/A | N/A |

In [ ]:
# ── Section 5a: Compute Pathway B for tasmin (frost days) ─────────────────────

print("Computing Pathway B for tasmin (frost days < 0°C)...")
hcr_b_tasmin = {ssp: {} for ssp in SCENARIOS}

for model in tqdm(MODELS, desc="tasmin Pathway B"):
    base_series = load_daily_series(model, "historical", "tasmin", BASELINE_YEARS)
    if base_series is None or len(base_series) < 5000:
        continue

    base_frost = count_frost_days(base_series)

    for ssp in SCENARIOS:
        fut_series = load_daily_series(model, ssp, "tasmin", FUTURE_YEARS)
        if fut_series is None or len(fut_series) < 5000:
            continue
        fut_frost = count_frost_days(fut_series)
        hcr_b = compute_hcr_pathway_b(base_frost, fut_frost)
        hcr_b_tasmin[ssp][model] = {
            "baseline_frost_days": base_frost,
            "future_frost_days": fut_frost,
            "hcr": hcr_b,
        }

for ssp in SCENARIOS:
    if hcr_b_tasmin[ssp]:
        hcrs = [r["hcr"] for r in hcr_b_tasmin[ssp].values()]
        print(f"\ntasmin Frost Days — Pathway B ({ssp}):")
        print(f"  Mean HCR: {np.mean(hcrs):+.4f}  ({np.mean(hcrs)*100:+.1f}%)")
        print(f"  → {'Fewer' if np.mean(hcrs) < 0 else 'More'} frost days (warming reduces cold extremes)")

In [ ]:
# ── Section 5b: Full HCR summary table — all variables ────────────────────────

HAZARD_MAP = {
    "tasmax": "Heat stress (heat wave days)",
    "tasmin": "Frost/freeze days",
    "tas":    "Thermal cycling (freeze-thaw)",
    "pr":     "Flood risk (extreme precip)",
    "sfcWind": "Structural fatigue",
    "hurs":   "Corrosion/mould",
    "rsds":   "N/A (resource, not hazard)",
}

summary_rows = []

for var in VARIABLES:
    for ssp in SCENARIOS:
        rc = report_cards.get(var, {}).get(ssp, {})
        conf = rc.get("tail_confidence", "N/A")
        mean_scvr = rc.get("mean_scvr", 0)

        # Pathway A (always available from SCVR)
        sf = SCALING_FACTORS.get(var, 0)
        hcr_a = mean_scvr * sf if var != "rsds" else None

        # Pathway B (only for variables we computed)
        hcr_b = None
        if var == "tasmax" and hcr_b_tasmax[ssp]:
            hcr_b = np.mean([r["hcr"] for r in hcr_b_tasmax[ssp].values()])
        elif var == "tasmin" and hcr_b_tasmin[ssp]:
            hcr_b = np.mean([r["hcr"] for r in hcr_b_tasmin[ssp].values()])
        elif var == "pr" and hcr_b_pr[ssp]:
            hcr_b = np.mean([r["hcr"] for r in hcr_b_pr[ssp].values()])

        # Select final HCR based on routing
        if var == "rsds":
            hcr_final = None
            pathway_used = "skip"
        elif conf == "DIVERGENT":
            hcr_final = hcr_b if hcr_b is not None else hcr_a
            pathway_used = "B only" if hcr_b is not None else "A (fallback)"
        elif conf in ("HIGH", "MODERATE"):
            if hcr_b is not None:
                # Use Pathway B as primary, A as cross-check
                hcr_final = hcr_b
                pathway_used = "A+B"
            else:
                hcr_final = hcr_a
                pathway_used = "A only"
        else:
            hcr_final = hcr_a
            pathway_used = "A only"

        summary_rows.append({
            "Variable": var,
            "Hazard": HAZARD_MAP.get(var, ""),
            "Scenario": ssp.upper(),
            "SCVR": f"{mean_scvr:+.4f}" if mean_scvr else "N/A",
            "Confidence": conf,
            "HCR_A": f"{hcr_a:+.4f}" if hcr_a is not None else "—",
            "HCR_B": f"{hcr_b:+.4f}" if hcr_b is not None else "—",
            "HCR Final": f"{hcr_final:+.4f}" if hcr_final is not None else "skip",
            "HCR %": f"{hcr_final*100:+.1f}%" if hcr_final is not None else "—",
            "Pathway": pathway_used,
        })

summary_df = pd.DataFrame(summary_rows)
print("═" * 95)
print("  FULL HCR RESULTS — All Variables × Scenarios")
print("═" * 95)
display(summary_df.style.hide(axis="index").set_properties(**{"text-align": "center"}))

In [ ]:
# ── Section 5c: Decade-level HCR for all variables + JSON output ──────────────

hcr_output = {
    "site": "hayhurst_solar",
    "generated": pd.Timestamp.now().isoformat(),
    "baseline_years": f"{BASELINE_YEARS[0]}-{BASELINE_YEARS[-1]}",
    "future_years": f"{FUTURE_YEARS[0]}-{FUTURE_YEARS[-1]}",
    "models": MODELS,
    "n_models": len(MODELS),
    "scaling_factors": SCALING_FACTORS,
    "epoch_hcr": {},
    "decade_hcr": {},
    "per_model_hcr": {},
    "pathway_b_details": {},
}

# Epoch-level HCR
for var in VARIABLES:
    hcr_output["epoch_hcr"][var] = {}
    for ssp in SCENARIOS:
        rc = report_cards.get(var, {}).get(ssp, {})
        mean_scvr = rc.get("mean_scvr", 0)
        sf = SCALING_FACTORS.get(var, 0)
        hcr_a = mean_scvr * sf if var != "rsds" else None

        hcr_b = None
        if var == "tasmax" and hcr_b_tasmax[ssp]:
            hcr_b = float(np.mean([r["hcr"] for r in hcr_b_tasmax[ssp].values()]))
        elif var == "tasmin" and hcr_b_tasmin[ssp]:
            hcr_b = float(np.mean([r["hcr"] for r in hcr_b_tasmin[ssp].values()]))
        elif var == "pr" and hcr_b_pr[ssp]:
            hcr_b = float(np.mean([r["hcr"] for r in hcr_b_pr[ssp].values()]))

        conf = rc.get("tail_confidence", "N/A")
        if var == "rsds":
            hcr_final = None
            pathway = "skip"
        elif conf == "DIVERGENT":
            hcr_final = hcr_b if hcr_b is not None else (hcr_a if hcr_a else None)
            pathway = "B" if hcr_b is not None else "A_fallback"
        elif hcr_b is not None:
            hcr_final = hcr_b
            pathway = "A+B"
        else:
            hcr_final = hcr_a
            pathway = "A"

        hcr_output["epoch_hcr"][var][ssp] = {
            "hcr_a": round(hcr_a, 6) if hcr_a is not None else None,
            "hcr_b": round(hcr_b, 6) if hcr_b is not None else None,
            "hcr_final": round(hcr_final, 6) if hcr_final is not None else None,
            "confidence": conf,
            "pathway_used": pathway,
        }

# Decade-level HCR (Pathway A for all, Pathway B where available)
for var in VARIABLES:
    if var == "rsds":
        continue
    hcr_output["decade_hcr"][var] = {}
    decomp = scvr_data.get(var, {})
    sf = SCALING_FACTORS.get(var, 0)

    for ssp in SCENARIOS:
        hcr_output["decade_hcr"][var][ssp] = {}
        decade_cards = decomp.get("decade_report_cards", {}).get(ssp, {})
        for dec_label in decade_labels:
            if dec_label in decade_cards:
                dec_scvr = decade_cards[dec_label]["mean_scvr"]
                hcr_a_dec = dec_scvr * sf
                hcr_output["decade_hcr"][var][ssp][dec_label] = {
                    "scvr": round(dec_scvr, 6),
                    "hcr_a": round(hcr_a_dec, 6),
                }

# Per-model Pathway B details (for tasmax, tasmin, pr)
for var_name, b_data in [("tasmax", hcr_b_tasmax), ("tasmin", hcr_b_tasmin), ("pr", hcr_b_pr)]:
    hcr_output["pathway_b_details"][var_name] = {}
    for ssp in SCENARIOS:
        hcr_output["pathway_b_details"][var_name][ssp] = {
            model: {k: (round(v, 6) if isinstance(v, float) else v)
                    for k, v in data.items()}
            for model, data in b_data[ssp].items()
        }

# Save JSON
output_path = OUTPUT_DIR / "hcr_results.json"
with open(output_path, "w") as f:
    json.dump(hcr_output, f, indent=2, default=str)
print(f"Saved: {output_path}")
print(f"  Size: {output_path.stat().st_size / 1024:.1f} KB")

# Save Parquet summary
parquet_rows = []
for var in VARIABLES:
    for ssp in SCENARIOS:
        epoch = hcr_output["epoch_hcr"].get(var, {}).get(ssp, {})
        for dec_label in decade_labels:
            decade = hcr_output.get("decade_hcr", {}).get(var, {}).get(ssp, {}).get(dec_label, {})
            parquet_rows.append({
                "variable": var,
                "scenario": ssp,
                "decade": dec_label,
                "scvr_epoch": epoch.get("hcr_a", None),
                "hcr_a": decade.get("hcr_a", epoch.get("hcr_a", None)),
                "hcr_b": epoch.get("hcr_b", None),
                "hcr_final": epoch.get("hcr_final", None),
                "confidence": epoch.get("confidence", None),
                "pathway_used": epoch.get("pathway_used", None),
            })

parquet_df = pd.DataFrame(parquet_rows)
parquet_path = OUTPUT_DIR / "hcr_summary.parquet"
parquet_df.to_parquet(parquet_path, index=False)
print(f"Saved: {parquet_path}")
display(parquet_df.head(12))

---

## Summary + Next Steps

**What this notebook produced:**
- HCR values for all 7 variables × 2 scenarios × 3 decades
- Pathway A (SCVR × scaling) and Pathway B (direct counting) for tasmax, tasmin, pr
- Cross-validation revealing implied scaling ≈ 26× for P90 thresholds (vs doc's 2.5× for absolute thresholds)
- Report Card routing validated: tasmax → A+B, pr → B only

**Key finding — scaling factor is threshold-dependent:**
- P90 per-DOY counting (strict climatological): ~26× amplification
- Absolute threshold counting (e.g., >43°C): ~2.5× amplification (per doc 07)
- **Recommendation: use Pathway B (direct counting) as the primary HCR method.** Pathway A is useful for quick screening but the scaling factor must be calibrated per threshold definition.

**Outputs saved:**
- `data/processed/hcr/hayhurst_solar/hcr_results.json` — full results with per-model Pathway B details
- `data/processed/hcr/hayhurst_solar/hcr_summary.parquet` — tabular summary

**What comes next (not in this notebook yet):**
- **EFR** (Part B) — Equipment Fatigue Ratio via Peck's model / Coffin-Manson
- **IUL** (Part C) — Impaired Useful Life from EFR
- **NAV** (Part D) — Net Asset Value impairment from HCR + IUL → CFADS adjustment